[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter14_learning_rates_and_schedules/notebook.ipynb)

# Chapter 14: Learning Rates and Schedules

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Every optimizer in Chapter 13 still needed one number from you. Momentum needed it, Adam needed it, and plain SGD needed it most of all. That number is the **learning rate**, and of everything you can set by hand it is the one that decides whether training works at all.

In June 2015 a researcher at the US Naval Research Laboratory in Washington published a short paper on the problem. Leslie Smith's subject was a scheme for varying the rate during training, but the part that spread fastest was a throwaway tool buried inside it. To find reasonable values, Smith suggested, do not run training once per candidate rate. Run it once, raising the rate as you go, and watch where the loss turns. One short run instead of a dozen long ones.

There is a small coincidence worth noticing. The perceptron in Chapter 2 was built for the US Navy in 1958, and the most useful practical trick for setting a learning rate came out of a Navy laboratory fifty seven years later.

This chapter is about choosing that number, and then about changing it while training runs. Before you read on, try to guess the answers to these four questions.

1. **What happens to training when the rate is far too small, and is it the same kind of failure as a rate that is far too large?**
2. **Could one short run tell you roughly which rates are worth trying?**
3. **Why might the best rate early in training be the wrong rate later?**
4. **If the loss becomes nan, what has actually happened to the weights?**

## The Most Important Hyperparameter

Start by measuring the thing rather than arguing about it. The cell below trains the same network on the same flowers in the same batches for the same thirty epochs, changing nothing but the learning rate.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

Everything below is Chapter 11's pipeline, reproduced exactly so that the learning rate is the only moving part in this chapter.

In [ ]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3,
    random_state=7, stratify=iris.target)
mean, std = X_train.mean(axis=0), X_train.std(axis=0)
Xtr = torch.tensor((X_train - mean) / std, dtype=torch.float32)
Xte = torch.tensor((X_test - mean) / std, dtype=torch.float32)
ytr = torch.tensor(y_train)
yte = torch.tensor(y_test)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
def make_net(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))

Seeding inside `make_net` matters here. Every run below must start from identical weights, or you would be measuring the starting point as much as the rate.

In [ ]:
def train(learning_rate, epochs=30, seed=0):
    net = make_net(seed)
    opt = torch.optim.SGD(net.parameters(), lr=learning_rate)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    history = []
    for epoch in range(epochs):
        total = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            batch_loss = loss_fn(net(xb), yb)
            batch_loss.backward()
            opt.step()
            total += batch_loss.item() * len(xb)
        history.append(total / len(Xtr))
    with torch.no_grad():
        accuracy = (net(Xte).argmax(1) == yte).float().mean().item()
    return history, accuracy

Now sweep across five orders of magnitude.

In [ ]:
for lr in [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 3.0, 10.0]:
    history, accuracy = train(lr)
    print(lr, round(history[-1], 4), round(accuracy, 3))

Read that from both ends. At 0.0001 the loss barely moves and the network ends up worse than guessing, not because anything broke but because thirty epochs of microscopic steps go almost nowhere. At 10.0 the loss is `nan`, which is not a large number but no number at all. In between there is a broad floor where the rate is good enough, and the whole useful range spans perhaps one factor of ten.

Figure 14.1 plots the same numbers with the rate on a log axis, which is the only sensible way to look at a quantity that matters across five orders of magnitude.

> 🖼️ **Figure 14.1:** Final training loss against learning rate, everything else held fixed. The curve falls steeply, flattens into a broad floor, and then turns back up before training breaks altogether just off the right of this axis.

The shape is the lesson. A rate that is ten times too small does not fail loudly, it just wastes your afternoon, and a beginner who sets 0.0001 and sees the loss sitting near 1.2 usually concludes that the network cannot learn the problem. A rate that is ten times too large fails immediately and obviously. Of the two, the quiet failure costs more time.

## Finding a Good Learning Rate

That sweep cost eight full training runs. Smith's trick costs about one.

The idea is to start from a rate so small that nothing happens, then multiply it by a little more than one after every batch, recording the loss as you go. Early on the rate is too small and the loss sits still. Then it enters the useful range and the loss falls. Then the rate grows past what the problem can take and the loss shoots up. You stop and look at where the fall happened.

In [ ]:
net = make_net(0)
opt = torch.optim.SGD(net.parameters(), lr=1e-5)
loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True,
                    generator=torch.Generator().manual_seed(0))
rates, losses = [], []
lr = 1e-5
stop = False
for epoch in range(30):
    if stop:
        break
    for xb, yb in loader:
        for group in opt.param_groups:
            group["lr"] = lr
        opt.zero_grad()
        batch_loss = loss_fn(net(xb), yb)
        batch_loss.backward()
        opt.step()
        rates.append(lr)
        losses.append(batch_loss.item())
        lr *= 1.15
        if batch_loss.item() > 8:
            stop = True
            break
print(len(rates), "steps")
print("lowest loss", round(min(losses), 4),
      "at rate", round(rates[losses.index(min(losses))], 4))

Note what is happening to `group["lr"]`. An optimizer keeps its settings in `param_groups`, and writing to that dictionary changes the rate for the next step. Every schedule in this chapter is doing exactly this underneath.

Those 104 steps are about fifteen epochs of work. The sweep in the previous section spent eight runs of thirty epochs, which is 240. The range test is roughly sixteen times cheaper and it searched a wider span.

> 🖼️ **Figure 14.2:** One run of the range test. The loss is flat while the rate is too small, falls through a band of useful rates, and climbs away once the rate is too large.

Now be careful about how much this tool actually tells you. The lowest point of the curve is not the rate to use, because by the time the loss is at its lowest the rate is already close to where the curve turns upward, and a rate that close to the edge is fragile. The usual advice is to take something inside the falling stretch, a few times below the lowest point.

Compare that advice against the sweep you already measured. The range test puts the band at roughly 0.2 to 1.5, and the sweep found its best result at 1.0, with 0.5 almost as good and 0.1 respectable. The tool did not hand you the single best value. What it did was take a search space spanning 0.0001 to 10 and cut it down to a band of about one order of magnitude, in one short run. That is what it is for, and expecting more from it leads to disappointment.

## Warmup and Decay Schedules

So far the rate has been fixed for the whole run. There is a good argument for changing it as training goes on, and the argument is easiest to see at the end of training rather than at the start.

A large step size is useful early, when the weights are random and far from anything sensible. Near the end the picture is different. The model is already close to a good setting, and a large step keeps throwing it past the target and back, so the loss stops improving and starts hopping around instead. Shrinking the rate late lets it settle.

PyTorch handles this with a **scheduler**, an object that sits beside the optimizer and adjusts the rate on a timetable. Two are common enough to learn by name.

In [ ]:
def train_scheduled(learning_rate, schedule=None, epochs=60, seed=0):
    net = make_net(seed)
    opt = torch.optim.SGD(net.parameters(), lr=learning_rate)
    sched = schedule(opt) if schedule else None
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    history = []
    for epoch in range(epochs):
        total = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            batch_loss = loss_fn(net(xb), yb)
            batch_loss.backward()
            opt.step()
            total += batch_loss.item() * len(xb)
        history.append(total / len(Xtr))
        if sched:
            sched.step()
    return history

step_decay = lambda o: torch.optim.lr_scheduler.StepLR(o, step_size=20, gamma=0.1)
cosine = lambda o: torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=60)

`StepLR` cuts the rate by a factor of `gamma` every `step_size` epochs, which here means dividing by ten at epoch 20 and again at epoch 40. `CosineAnnealingLR` eases the rate down a smooth curve to nearly zero over `T_max` epochs. Figure 14.3 draws both.

> 🖼️ **Figure 14.3:** Two ways to bring the rate down over sixty epochs. One drops in stages, the other glides.

Now run all three for sixty epochs from the same starting rate.

In [ ]:
for name, schedule in [("constant", None), ("step", step_decay), ("cosine", cosine)]:
    history = train_scheduled(1.0, schedule)
    tail = history[-10:]
    print(name, round(history[-1], 4),
          round(min(tail), 4), round(max(tail), 4))

This result deserves more care than it usually gets, because the first column says the schedules lost. The constant rate ended on the lowest loss of the three.

Look at the other two columns. Over its last ten epochs the constant run wandered between 0.0183 and 0.0757, a spread of more than four times. The cosine run sat between 0.0262 and 0.0271, a spread of less than four percent. The constant run's final figure of 0.0183 was the best of its last ten epochs; stopping one epoch earlier or later would have reported something considerably worse.

So the schedule did not buy a lower number here. It bought a *settled* one. Figure 14.4 shows the last twenty epochs of both.

> 🖼️ **Figure 14.4:** The last twenty epochs of the same two runs. The constant rate is still hopping about; the decayed one has stopped moving.

That distinction matters more than it looks. A result you can reproduce is worth more than a slightly better result you got by stopping on a lucky epoch, and Book 5 made the same argument about evaluation.

The other half of this section is **warmup**, which is the opposite move: start below your chosen rate and climb to it over the first few epochs. The reasoning is that the first steps are taken from random weights, where the slopes can be unusually large and a full sized step can wreck the model before it has learned anything.

Here is the honest result. On this problem warmup does nothing measurable. Running the flowers at a peak rate of 4.0 with and without a five epoch warmup gives final losses that differ in both directions depending on the seed, and the rate of 10.0 that exploded earlier still explodes, just a little later. Warmup is not a rescue for a rate that is simply too large.

Where warmup does earn its place is on large models trained with large batches, where the first few hundred steps really can be destructive and the published recipes for those models almost all include it. You will meet it again in Book 8. On a network with 131 parameters and 105 flowers there is nothing for it to protect.

## When Training Diverges

The `nan` in the first table is worth understanding rather than merely avoiding. Watch what happens on the way there.

In [ ]:
history, accuracy = train(10.0, epochs=6)
for value in history:
    print("%.4g" % value)

Nothing mysterious occurred. The step was large enough that each update overshot the valley and landed further up the opposite side than it started, so the next slope was steeper, so the next step was larger still. The loss gains several orders of magnitude every epoch until the numbers exceed what a float32 can hold, which prints as `inf`, and the first arithmetic on an infinity that has no answer, such as infinity minus infinity, produces `nan`. After that every weight touched by that gradient is `nan` and the model is dead. There is no recovering a run from this point; you lower the rate and start again.

There is a second cause of the same symptom that has nothing to do with the rate you chose, and Chapter 11 promised to come back to it. It is the scale of your inputs.

In [ ]:
Xtr_raw = torch.tensor(X_train, dtype=torch.float32)
print([round(v, 2) for v in Xtr_raw.mean(0).tolist()])
print([round(v, 2) for v in Xtr_raw.std(0).tolist()])

The raw measurements have means from 1.21 to 5.89 and spreads from 0.45 to 1.82, so one petal width column varies four times less than one petal length column. A weight on the wide column feels a much larger slope than a weight on the narrow one, which is the ravine from Chapter 13 again, except now you built it yourself by not scaling.

In [ ]:
def train_raw(learning_rate, epochs=30, seed=0):
    net = make_net(seed)
    opt = torch.optim.SGD(net.parameters(), lr=learning_rate)
    loader = DataLoader(TensorDataset(Xtr_raw, ytr), batch_size=16, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    for epoch in range(epochs):
        total = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            batch_loss = loss_fn(net(xb), yb)
            batch_loss.backward()
            opt.step()
            total += batch_loss.item() * len(xb)
    return total / len(Xtr_raw)

for lr in [1.0, 0.1, 0.01, 0.001]:
    print(lr, round(train_raw(lr), 4))

On scaled features a rate of 1.0 reached 0.034. On the same data unscaled, the best of those four rates reaches 0.1762, five times worse, and 1.0 now barely trains at all. Scaling your inputs is not a tidying habit. It is what makes a single learning rate appropriate for every weight at once, and skipping it costs you more than any optimizer choice in Chapter 13 can win back.

Table 14.1 collects the symptoms worth recognising.

**Table 14.1: Reading a failing run. The first four are all the learning rate in different disguises, which is why it deserves the first look.**

| What you see | Most likely cause | What to try |
|---|---|---|
| Loss becomes nan or inf | Rate far too large, or unscaled inputs | Divide the rate by ten, scale the features |
| Loss grows steadily | Rate too large | Divide the rate by three to ten |
| Loss hops about near the end | Rate fine early, too large late | Add a decay schedule |
| Loss falls far too slowly | Rate too small | Run the range test |
| Loss stuck at the baseline | Rate tiny, or a deeper fault | Chapter 16 |

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Change the multiplier.** Run the range test with `lr *= 1.05` instead of 1.15. It takes more steps; check whether the band it suggests is the same.
2. **Test the settling claim.** Run the constant and cosine versions at a starting rate of 3.0 and compare the spread of the last ten epochs, not just the final number.
3. **Find the breaking point.** Narrow down where training stops surviving, between a rate of 3.0 and 10.0, to within half a unit.
4. **Scale one column only.** Standardise three of the four iris columns and leave the widest one raw, then see how much of the damage that single column accounts for.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **Why is a rate that is too small more expensive in practice than one that is too large?**
2. **What does the range test actually give you, and what does it not give you?**
3. **Why should a learning rate sweep be plotted on a log axis?**
4. **A schedule produced a higher final loss than a constant rate. Why might you still use it?**
5. **How does a nan loss come about, step by step?**
6. **What does scaling the inputs have to do with the learning rate?**

### Answers

1. Because it fails quietly. A diverging run announces itself within seconds, while a crawling run looks like a network that cannot learn the problem, and people abandon the model rather than the rate.
2. It gives you a band of roughly one order of magnitude that is worth trying, for about the cost of one training run. It does not give you the single best value, and the lowest point of its curve is already too close to the edge to use.
3. Because the useful range spans several factors of ten. On a linear axis every rate below 1.0 would be squashed against the left edge.
4. Because the constant run's final loss was the luckiest of its last ten epochs and the decayed run's was stable. A reproducible result is worth more than a slightly better unreproducible one.
5. A step overshoots and lands higher than it started, so the next slope is steeper and the next step larger. The loss multiplies each epoch until it passes what float32 can hold, giving inf, and then an operation like infinity minus infinity gives nan, which spreads to every weight.
6. A single rate is applied to every weight. If one input column is much wider than another, the weights on it feel much larger slopes, so no single rate suits both at once. Scaling removes the mismatch.

## What Comes Next

You can now choose a learning rate with a measurement rather than a guess, read the shape of a sweep, use a schedule for the reason it actually helps, and recognise the four failures that are really the rate in disguise.

- **Chapter 15, Overfitting and Regularization,** takes on the loss curve from Chapter 11's table that falls on training data while rising on held out data.
- **Chapter 16, Making Training Stable,** handles the last row of this chapter's table, where the loss sits at the baseline and the learning rate is not the reason.

Chapter 15 starts by making a network memorise its training set on purpose, so you can watch overfitting happen rather than read about it.